# Sesión 38 · Fine-tuning con Trainer y Evaluate

Ajusta un modelo preentrenado a tu tarea sin escribir el bucle de entrenamiento.

**Objetivos:**
1. Preparar dataset y tokenizador para el `Trainer`.
2. Afinar un modelo binario (sentimiento) con `Trainer` + `evaluate`.
3. Guardar el modelo y usarlo como pipeline.
4. Afinar un modelo multiclase (Yelp 5 estrellas).
5. Publicar el modelo en el Hub.

> **Formato:** ~30 min explicación · ~75 min notebook · ~15 min reto.

In [ ]:
%pip install -q transformers datasets evaluate accelerate matplotlib torch

---
## 0 · Preparación: dataset, tokenizador y dispositivo

Cargamos el dataset IMDB, lo tokenizamos y detectamos si hay GPU disponible.
Estos objetos (`train`, `test`, `tokenizer`, `device`) se usan en todas las secciones.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# ── Dataset ──────────────────────────────────────────────────────────────
dataset   = load_dataset("stanfordnlp/imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenizar(ejemplos):
    return tokenizer(
        ejemplos["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train = dataset["train"].shuffle(seed=42).select(range(1000))
test  = dataset["test"].shuffle(seed=42).select(range(200))
train = train.map(tokenizar, batched=True)
test  = test.map(tokenizar, batched=True)

# ── Dispositivo ──────────────────────────────────────────────────────────
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Dataset listo. Dispositivo: {device}")

---
## 1 · Fine-tuning binario con `Trainer`

`TrainingArguments` controla hiperparámetros. `Trainer` orquesta el bucle de
entrenamiento, evaluación y guardado.

In [ ]:
import torch
import evaluate
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# Detección de dispositivo (Colab puede tener GPU o CPU)
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Dispositivo: {device}")

modelo = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(device)

accuracy = evaluate.load("accuracy")

def metricas(eval_pred):
    logits, etiquetas = eval_pred
    predicciones = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predicciones, references=etiquetas)

args = TrainingArguments(
    output_dir="./resultados",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,          # 1 época para Colab; usa 3 en producción
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=50,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=train,
    eval_dataset=test,
    compute_metrics=metricas,
)

trainer.train()
print(trainer.evaluate())

---
## 2 · Guardar y reutilizar el modelo

In [ ]:
# Guardar modelo y tokenizer juntos
modelo.save_pretrained("./mi_modelo")
tokenizer.save_pretrained("./mi_modelo")

# Reutilizar como pipeline
from transformers import pipeline

clasificador_local = pipeline("sentiment-analysis", model="./mi_modelo")
print(clasificador_local("Me ha encantado el producto, llegó antes de tiempo."))
print(clasificador_local("Fue un desastre, llegó roto y tarde."))

In [ ]:
# Publicar en el Hub (requiere token de escritura: hf auth login)
# trainer.push_to_hub("tu-usuario/mi-modelo-sentimiento")
# O bien:
# modelo.push_to_hub("tu-usuario/mi-modelo-sentimiento")
# tokenizer.push_to_hub("tu-usuario/mi-modelo-sentimiento")
print("Descomenta las líneas anteriores con tu usuario del Hub.")

---
## 3 · Fine-tuning multiclase (Yelp, 5 estrellas)

Mismo flujo, pero con `num_labels=5` y el dataset `yelp_review_full`.
El modelo aprenderá a predecir puntuaciones de 1 a 5 estrellas.

In [ ]:
ds_yelp = load_dataset("yelp_review_full")

# Solo reseñas de restaurantes para reducir el tamaño
def es_restaurante(x):
    return "restaurant" in x["text"].lower()

train_y = ds_yelp["train"].filter(es_restaurante).shuffle(seed=42).select(range(2000))
test_y  = ds_yelp["test"].filter(es_restaurante).shuffle(seed=42).select(range(400))

train_y = train_y.map(tokenizar, batched=True)
test_y  = test_y.map(tokenizar, batched=True)

modelo_multi = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=5
).to(device)

args_multi = TrainingArguments(
    output_dir="./resultados_multi",
    eval_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    logging_steps=50,
)

trainer_multi = Trainer(
    model=modelo_multi,
    args=args_multi,
    train_dataset=train_y,
    eval_dataset=test_y,
    compute_metrics=metricas,
)

trainer_multi.train()
print(trainer_multi.evaluate())

In [ ]:
# Inferencia manual con softmax → estrellas
modelo_multi.eval()
reseñas = [
    "Amazing food and excellent service, will definitely come back!",
    "Terrible experience, cold food and rude staff.",
    "Decent place, nothing special but okay.",
]

inputs_res = tokenizer(reseñas, padding=True, truncation=True, return_tensors="pt").to(device)

with torch.no_grad():
    logits_res = modelo_multi(**inputs_res).logits

estrellas = torch.argmax(logits_res, dim=-1) + 1  # clases 0-4 → 1-5
for reseña, n_estrellas in zip(reseñas, estrellas):
    print(f"{'★' * n_estrellas.item():5s}  {reseña[:60]}")

---
## Práctica

Ajusta el modelo sobre el dataset **en español** `dccuchile/bert-base-spanish-wwm-uncased`
y compara la accuracy con el modelo en inglés sobre las mismas reseñas.

1. Carga un dataset de reseñas en español (p. ej. busca `reseñas` o `reviews_es` en el Hub).
2. Tokeniza con el tokenizer del modelo español.
3. Entrena 1 época.
4. Compara accuracy con el modelo inglés fine-tuneado en la sección 1.

In [ ]:
# TODO: fine-tuning en español y comparación

---
## Reto · NER o clasificación multietiqueta

**Opción A — NER:** Afina un modelo `AutoModelForTokenClassification` sobre
el dataset `conll2003` para reconocimiento de entidades en inglés.

**Opción B — Multietiqueta:** Usando el dataset `zeroshot/twitter-financial-news-topic`
y un modelo con `num_labels=20`, entrena un clasificador de tópicos financieros.
Muestra la distribución de errores por categoría.

In [ ]:
# TODO: NER o clasificación multietiqueta